In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.environ['MYSQL_USER']}:{os.environ['MYSQL_PWD']}"
    f"@{os.environ['MYSQL_HOST']}/{os.environ['MYSQL_DB']}")
SNAPSHOT = pd.Timestamp("2026-09-30")   # report date used when generating the data
log = {}

In [3]:
raw = pd.read_sql("SELECT * FROM stg_claims", engine)
log["raw rows"] = len(raw)
raw.info()
raw.head()

<class 'pandas.DataFrame'>
RangeIndex: 9172 entries, 0 to 9171
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   claim_id            9172 non-null   str  
 1   patient_id          9126 non-null   str  
 2   payer_name          9172 non-null   str  
 3   provider_name       9172 non-null   str  
 4   cpt_code            9172 non-null   str  
 5   dos_date            9172 non-null   str  
 6   submit_date         9172 non-null   str  
 7   payment_date        8411 non-null   str  
 8   billed_amount       9172 non-null   str  
 9   allowed_amount      9172 non-null   str  
 10  paid_amount         9172 non-null   str  
 11  patient_resp        9172 non-null   str  
 12  claim_status        9172 non-null   str  
 13  carc_code           995 non-null    str  
 14  is_first_pass_paid  9172 non-null   str  
dtypes: str(15)
memory usage: 1.0 MB


,claim_id,patient_id,payer_name,provider_name,cpt_code,dos_date,submit_date,payment_date,billed_amount,allowed_amount,paid_amount,patient_resp,claim_status,carc_code,is_first_pass_paid
0,CLM004475,P00503,Cigna,Dr. David Okafor,99214,2025-01-04,2025-01-08,2025-02-06,195.0,124.59,99.67,24.92,Paid,NaN,1
1,CLM001025,P00257,Medicare,Dr. Michael Brooks,99204,2025-01-14,2025-01-28,2025-02-13,290.0,108.71,108.71,0.0,Paid,NaN,1
2,CLM000527,P01443,Medicare Part B,Dr. Sarah Mitchell,99395,2025-05-25,2025-09-02,2025-09-20,220.0,82.8,66.24,16.56,Paid,NaN,1
3,CLM001818,P00066,Cigna,Dr. Emily Tran,99213,2025-04-11,2025-04-22,2025-05-20,135.0,78.74,62.99,15.75,Paid,NaN,1
4,CLM004054,P01325,Blue Cross Blue Shield TX,Dr. Priya Nair,71046,2025-11-17,2025-11-28,2025-12-24,110.0,65.68,65.68,0.0,Paid,NaN,1


In [ ]:
#the problems before cleaning
print("exact duplicate rows:", raw.duplicated().sum())
print("distinct payer names:", raw.payer_name.nunique())
print("cpt codes with spaces:", (raw.cpt_code != raw.cpt_code.str.strip()).sum())
print("US-format dates:", raw.dos_date.str.contains("/").sum())
print(raw.isna().sum()[lambda s: s > 0])
raw.payer_name.value_counts()

exact duplicate rows: 136
distinct payer names: 26
cpt codes with spaces: 276
US-format dates: 459
patient_id        46
payment_date     761
carc_code       8177
dtype: int64


payer_name
Medicare                      1818
UnitedHealthcare              1626
Aetna                         1127
Blue Cross Blue Shield TX     1120
Medicaid TX                    961
Cigna                          933
Humana                         490
MEDICARE                       130
Medicare Part B                122
UHC                             80
UNITED HEALTHCARE               80
United Healthcare               76
Cigna Healthcare                65
CIGNA                           65
AETNA                           60
aetna                           54
Aetna Inc                       46
blue cross blue shield tx       40
MEDICAID                        38
BCBS TX                         38
Medicaid Texas                  37
BCBS of Texas                   36
TX Medicaid                     34
HUMANA                          34
Humana Inc.                     33
Blue Cross TX                   29
Name: count, dtype: int64

In [ ]:
#trim, deduplicate, standardize payers
df = raw.copy()
for c in df.columns:
    df[c] = df[c].str.strip()

n = len(df)
df = df.drop_duplicates().reset_index(drop=True)
log["duplicate rows removed"] = n - len(df)

def canon_payer(name):
    n = str(name).lower()
    if "medicaid" in n: return "Medicaid TX"      # check before "medicare"
    if "medicare" in n: return "Medicare"
    if "bcbs" in n or "blue cross" in n: return "Blue Cross Blue Shield TX"
    if "united" in n or "uhc" in n: return "UnitedHealthcare"
    if "aetna" in n: return "Aetna"
    if "cigna" in n: return "Cigna"
    if "humana" in n: return "Humana"
    return None

df["payer_name"] = df["payer_name"].map(canon_payer)
print(df.payer_name.value_counts(dropna=False))   

payer_name
Medicare                     2040
UnitedHealthcare             1836
Aetna                        1268
Blue Cross Blue Shield TX    1244
Medicaid TX                  1053
Cigna                        1050
Humana                        545
Name: count, dtype: int64


In [8]:
def parse_dates(s):
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    us = pd.to_datetime(s, format="%m/%d/%Y", errors="coerce")
    return iso.fillna(us)

for c in ["dos_date", "submit_date", "payment_date"]:
    df[c] = parse_dates(df[c])
for c in ["billed_amount", "allowed_amount", "paid_amount", "patient_resp"]:
    df[c] = pd.to_numeric(df[c])
df["is_first_pass_paid"] = pd.to_numeric(df["is_first_pass_paid"]).astype(int)

# every non-null date should have parsed
print(df[["dos_date", "submit_date", "payment_date"]].isna().sum())

dos_date          0
submit_date       0
payment_date    752
dtype: int64


In [9]:
bad_paid = df["paid_amount"] > df["allowed_amount"]
df.loc[bad_paid, "paid_amount"] = (df["allowed_amount"] - df["patient_resp"]).round(2)
log["paid > allowed, recomputed"] = int(bad_paid.sum())

miss_pat = df["patient_id"].isna()
df.loc[miss_pat, "patient_id"] = "UNKNOWN"
log["missing patient_id set to UNKNOWN"] = int(miss_pat.sum())

log["paid, no payment date (kept)"] = int(
    ((df["claim_status"] == "Paid") & df["payment_date"].isna()).sum())

In [10]:
dim_payer = pd.read_sql("SELECT payer_key, payer_name FROM dim_payer", engine)
dim_provider = pd.read_sql("SELECT provider_key, provider_name FROM dim_provider", engine)
dim_procedure = pd.read_sql("SELECT procedure_key, cpt_code FROM dim_procedure", engine)

df = (df.merge(dim_payer, on="payer_name", how="left")
        .merge(dim_provider, on="provider_name", how="left")
        .merge(dim_procedure, on="cpt_code", how="left"))

rules = {
    "service date after submit date": df["dos_date"] > df["submit_date"],
    "date after report date": (df["submit_date"] > SNAPSHOT) | (df["payment_date"] > SNAPSHOT),
    "unmapped payer/provider/cpt": df[["payer_key", "provider_key", "procedure_key"]].isna().any(axis=1),
}
reasons = pd.Series("", index=df.index)
for msg, mask in rules.items():
    reasons = reasons.where(~mask, reasons + msg + "; ")
    log[f"quarantined: {msg}"] = int(mask.sum())

rejected = df[reasons != ""].copy()
rejected["reject_reason"] = reasons[reasons != ""]
good = df[reasons == ""].copy()
len(good), len(rejected)

(8914, 122)

In [11]:
for src, dst in [("dos_date", "dos_date_key"), ("submit_date", "submit_date_key"),
                 ("payment_date", "payment_date_key")]:
    good[dst] = good[src].dt.strftime("%Y%m%d").astype("Int64")
for k in ["payer_key", "provider_key", "procedure_key"]:
    good[k] = good[k].astype(int)

fact = good[["claim_id", "patient_id", "payer_key", "provider_key", "procedure_key",
             "dos_date_key", "submit_date_key", "payment_date_key", "billed_amount",
             "allowed_amount", "paid_amount", "patient_resp", "claim_status",
             "carc_code", "is_first_pass_paid"]]

for c in ["dos_date", "submit_date", "payment_date"]:
    rejected[c] = rejected[c].dt.strftime("%Y-%m-%d")

with engine.begin() as conn:
    conn.execute(text("DELETE FROM fact_claim_line"))
fact.to_sql("fact_claim_line", engine, if_exists="append", index=False)
rejected.to_sql("rejected_claims", engine, if_exists="replace", index=False)
log["rows loaded to fact_claim_line"] = len(fact)

In [12]:
print(pd.read_sql("SELECT COUNT(*) AS fact_rows FROM fact_claim_line", engine))
print(pd.read_sql("SELECT claim_status, COUNT(*) n FROM fact_claim_line GROUP BY claim_status", engine))
print(pd.read_sql("SELECT reject_reason, COUNT(*) n FROM rejected_claims GROUP BY reject_reason", engine))
for k, v in log.items():
    print(f"{k}: {v}")

   fact_rows
0       8914
  claim_status     n
0         Paid  8349
1  Written-off   259
2     Appealed   149
3       Denied   157
                                       reject_reason   n
0                   service date after submit date;   71
1                           date after report date;   50
2  service date after submit date; date after rep...   1
raw rows: 9172
duplicate rows removed: 136
paid > allowed, recomputed: 42
missing patient_id set to UNKNOWN: 45
paid, no payment date (kept): 169
quarantined: service date after submit date: 72
quarantined: date after report date: 51
quarantined: unmapped payer/provider/cpt: 0
rows loaded to fact_claim_line: 8914


In [13]:
assert len(fact) + len(rejected) == log["raw rows"] - log["duplicate rows removed"], "rows lost"
assert not fact.duplicated().any(), "duplicates remain"
assert fact[["payer_key", "provider_key", "procedure_key", "dos_date_key", "submit_date_key"]].notna().all().all(), "null keys"
assert fact.payer_key.nunique() == 7, "payer count wrong"
assert (good.dos_date <= good.submit_date).all(), "dates out of order"
assert (good.paid_amount <= good.allowed_amount).all(), "paid > allowed"
assert ((good.paid_amount + good.patient_resp - good.allowed_amount).abs() < 0.015).all(), "billing identity broken"
print("all checks passed")

all checks passed


In [14]:
summary = pd.DataFrame({
    "before (raw)": [len(raw), raw.payer_name.nunique(), raw.duplicated().sum(),
                     int((raw.cpt_code != raw.cpt_code.str.strip()).sum())],
    "after (clean)": [len(fact), good.payer_name.nunique(), int(fact.duplicated().sum()), 0],
}, index=["rows", "distinct payer names", "duplicate rows", "cpt codes with spaces"])
summary

,before (raw),after (clean)
rows,9172,8914
distinct payer names,26,7
duplicate rows,136,0
cpt codes with spaces,276,0
